In [2]:
import pandas as pd
import numpy as np

# 1. Load Dataset Final
try:
    df_final = pd.read_csv('../../datas/dataset_final.csv')
    print(f"Data Loaded: {df_final.shape}")
except:
    print("File dataset_final.csv tidak ditemukan. Pastikan path-nya benar.")

# 2. Siapkan "Database Profil User"
# Kita butuh daftar unik setiap user beserta atribut fisiknya untuk dicocokkan
# Kita pakai kolom dengan akhiran _x karena itu biasanya data profil awal
cols_profile = [
    'User_ID', 'Age_x', 'Gender_x', 'Height_cm_x', 'Initial_Weight_kg_x', 
    'Goal_x', 'Workout_Frequency_x', 'level_x'
]

# Ambil 1 baris per user saja (Drop duplikat karena User_ID berulang di dataset_final)
df_users_db = df_final[cols_profile].drop_duplicates(subset=['User_ID']).copy()

print(f"Jumlah User Unik di Database: {len(df_users_db)}")

# --- FUNGSI 1: CARI KEMBARAN (SIMILARITY) ---
def find_similar_user(new_user_data):
    """
    Mencari User_ID dari database yang paling mirip dengan user baru.
    """
    candidates = df_users_db.copy()
    
    # FILTER 1: Cari yang Goal-nya SAMA (Wajib)
    # Kalau goal beda (misal user mau Muscle Gain tapi dikasih Weight Loss), jadwalnya pasti salah.
    candidates = candidates[candidates['Goal_x'] == new_user_data['Goal']]
    
    # FILTER 2: Cari yang Frekuensinya SAMA (Wajib)
    # Biar jadwalnya pas (misal user minta 3 hari, dikasih jadwal 3 hari)
    candidates = candidates[candidates['Workout_Frequency_x'] == new_user_data['Workout_Frequency']]
    
    # Jika tidak ada yang cocok persis, kita longgarkan filter (ambil Goal aja)
    if candidates.empty:
        print("Note: Tidak menemukan frekuensi yang sama persis, mencari berdasarkan Goal saja...")
        candidates = df_users_db[df_users_db['Goal_x'] == new_user_data['Goal']].copy()
    
    # FILTER 3: Hitung Skor Kemiripan (Similarity Score)
    # Semakin kecil skor, semakin mirip.
    # Kita bandingkan Umur, Tinggi, dan Berat.
    
    # Kita ubah Gender jadi angka dulu (0/1) buat hitungan
    gender_input = 1 if new_user_data['Gender'] == 'Male' else 0
    candidates['Gender_Num'] = candidates['Gender_x'].apply(lambda x: 1 if x == 'Male' else 0)
    
    candidates['Similarity_Score'] = (
        abs(candidates['Age_x'] - new_user_data['Age']) * 1 +            # Bobot Umur: 1
        abs(candidates['Height_cm_x'] - new_user_data['Height']) * 2 +   # Bobot Tinggi: 2
        abs(candidates['Initial_Weight_kg_x'] - new_user_data['Weight']) * 2 + # Bobot Berat: 2
        abs(candidates['Gender_Num'] - gender_input) * 50                # Bobot Gender: 50 (Harus sama!)
    )
    
    # Urutkan dari skor terkecil (paling mirip)
    best_match = candidates.sort_values('Similarity_Score').iloc[0]
    
    return best_match['User_ID'], best_match['Similarity_Score']

# --- FUNGSI 2: AMBIL JADWAL ---
def get_workout_plan(user_id_target):
    """
    Mengambil semua jadwal latihan milik User ID tertentu.
    """
    # Ambil baris data milik user tersebut
    schedule = df_final[df_final['User_ID'] == user_id_target].copy()
    
    # Pilih kolom yang mau ditampilkan sebagai jadwal
    cols_display = [
        'Day', 'Muscle Group', 'Exercise Name', 
        'Equipment', 'Sets', 'Reps', 'Instructions'
    ]
    
    # Rapikan: Urutkan berdasarkan Hari, lalu Buang duplikat latihan (kalau ada)
    # Kita asumsikan kolom 'Day' bisa diurutkan string-nya (Day 1, Day 2...)
    schedule_clean = schedule[cols_display].drop_duplicates()
    
    return schedule_clean

# --- CONTOH PEMAKAIAN ---

# 1. Data User Baru (Input dari Frontend/Aplikasi)
new_user = {
    'Age': 25,
    'Gender': 'Male',
    'Height': 175,
    'Weight': 70,
    'Goal': 'Muscle Gain',        # Harus sama persis tulisannya dengan di CSV
    'Workout_Frequency': 4,       # Mau latihan 4 hari
    'Level': 'Beginner'
}

print("Mencari rekomendasi untuk user baru...")
print(f"Profil: {new_user}")

# 2. Cari ID User yang mau "dicontek"
matched_id, score = find_similar_user(new_user)
print(f"\n✅ Ditemukan Kembaran! User ID: {matched_id} (Skor Beda: {score})")
print("Mengambil jadwal latihan dari User ID tersebut...\n")

# 3. Tampilkan Jadwalnya
rekomendasi_jadwal = get_workout_plan(matched_id)

# Menampilkan per Hari biar rapi
unique_days = rekomendasi_jadwal['Day'].unique()
sorted_days = sorted(unique_days) # Mengurutkan Day 1, Day 2, dst

for day in sorted_days:
    print(f"📅 {day}")
    day_plan = rekomendasi_jadwal[rekomendasi_jadwal['Day'] == day]
    # Tampilkan kolom penting saja biar ga kepanjangan
    display(day_plan[['Muscle Group', 'Exercise Name', 'Sets', 'Reps', 'Equipment']])
    print("-" * 50)

Data Loaded: (21853, 61)
Jumlah User Unik di Database: 100
Mencari rekomendasi untuk user baru...
Profil: {'Age': 25, 'Gender': 'Male', 'Height': 175, 'Weight': 70, 'Goal': 'Muscle Gain', 'Workout_Frequency': 4, 'Level': 'Beginner'}

✅ Ditemukan Kembaran! User ID: 51 (Skor Beda: 25)
Mengambil jadwal latihan dari User ID tersebut...

📅 Day 1 - Upper A


,Muscle Group,Exercise Name,Sets,Reps,Equipment
10270,Chest,barbell incline bench press,3,8-12,barbell
10271,Back,weighted hyperextension (on stability ball),3,8-12,weighted
10272,Shoulders,dumbbell front raise,3,8-12,dumbbell


--------------------------------------------------
📅 Day 2 - Lower A


,Muscle Group,Exercise Name,Sets,Reps,Equipment
10273,Legs,barbell standing calf raise,3,8-12,barbell
10274,Legs,hack calf raise,3,8-12,sled machine
10275,Abs,assisted hanging knee raise with throw down,3,8-12,assisted
10276,Abs,weighted side bend (on stability ball),3,8-12,weighted


--------------------------------------------------
📅 Day 3 - Upper B


,Muscle Group,Exercise Name,Sets,Reps,Equipment
10277,Biceps,cable seated curl,3,8-12,cable
10278,Triceps,dumbbell lying single extension,3,8-12,dumbbell
10279,Chest,smith incline bench press,3,8-12,smith machine


--------------------------------------------------
📅 Day 4 - Lower B


,Muscle Group,Exercise Name,Sets,Reps,Equipment
10280,Legs,hack calf raise,3,8-12,sled machine
10281,Legs,barbell standing rocking leg calf raise,3,8-12,barbell
10282,Abs,weighted side bend (on stability ball),3,8-12,weighted
10283,Abs,cable seated crunch,3,8-12,cable


--------------------------------------------------
